# Quickstart : Querying PDF with Astra and LanChain

A question answering demo using Astra DB and LangChain, powered by Vector Search

## Prerequisites:

You need a **Serverless Cassandra with Vector Search** database on [Astra Db](https://www.datastax.com/products/datastax-astra) to run this demo. As outlined in more details, you will need a DB token with role *Database Administrator* and copy your Database ID. these connection parameters are need to create a vector database.

What  will you do?
* Setup: import dependencies, provide secrets and create the langchain vector store.
* Run a Question Answer retrieving the relavant headlines and having the LLM construct the answers.

In [1]:
# Load in environment variables
import os
from dotenv import load_dotenv

load_dotenv()


os.environ["LANGSMITH_API_KEY"] = os.environ["LANGCHAIN_API_KEY"]
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") # use groq for llm
api_endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
db_token= os.getenv("ASTRA_DB_APPLICATION_TOKEN")

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_astradb import AstraDBVectorStore
from langchain_groq import ChatGroq

### Load the PDF

In [3]:
# load and split the pdf into chunk documents 
loader=  PyPDFDirectoryLoader("research_papers")

# load the docs
docs = loader.load()

In [4]:
len(docs)

61

### Split the docs into chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

splits = splitter.split_documents(docs)

len(splits)

195

In [6]:
splits[0]

Document(metadata={'source': 'research_papers/Attention.pdf', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network archite

### Create an embedding function

We will use opensource HuggingFace Embeddings

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

### Create vector store and retriever

#### Avoiding duplicate uploads

In [8]:
# --- Prepare documents with unique IDs based on content ---
import hashlib
from langchain_core.documents import Document


docs_with_ids = []
for doc in splits:
    content = doc.page_content if isinstance(doc, Document) else doc
    doc_id = hashlib.md5(content.encode("utf-8")).hexdigest()  # unique hash of content
    docs_with_ids.append(
        Document(page_content=content, metadata={"id": doc_id})
    )

# --- Initialize AstraDB vector store ---
vector_store = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="astra_langchain_rag",
    api_endpoint=api_endpoint,
    token=db_token
)

# --- Upload documents safely ---
for doc in docs_with_ids:
    vector_store.add_texts([doc.page_content], ids=[doc.metadata["id"]])

print("Documents uploaded successfully without duplicates!")

Documents uploaded successfully without duplicates!


In [9]:
# check the db by performing similarity search
vector_store.similarity_search("what is attention?", search_kwargs={"k": 2})

[Document(page_content='Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nFigure 3: An example of the attention mechanism following long-distance dependencies in the\nencoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of\nthe verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for\nthe word ‘making’. Different colors represent different heads. Best viewed in color.\n13'),
 Document(page_content='Attention in transformers [64] calculates query, key, and value\

It's working, let's move to the next part, converting it into retreiver

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

### Create prompts and create rag chain

In [11]:
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
    Answer the questions based on the provided context only.
    Please provide the most accurate response on the queston
    <context>
    {context}
    <context>
    Question:{input}
    """)


llm = ChatGroq(model_name="gemma2-9b-it")

output_parser = StrOutputParser()

In [12]:
documents_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    output_parser=output_parser
)

rag_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=documents_chain
)

response = rag_chain.invoke({"input": "what is attention?"})
print(response["answer"])   # usually "answer" key in response


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Attention in transformers calculates query, key, and value mappings for input sequences. The attention score is obtained by multiplying the query and key, and later used to weight values.  




In [13]:
response['context']

[Document(page_content='Attention in transformers [64] calculates query, key, and value\nmappings for input sequences, where the attention score is\nobtained by multiplying the query and key, and later used to\nweight values. We discuss different attention strategies used in\nLLMs below.\nSelf-Attention [64]: Calculates attention using queries, keys,\nand values from the same block (encoder or decoder).\nCross Attention: It is used in encoder-decoder architectures,\nwhere encoder outputs are the queries, and key-value pairs\ncome from the decoder.\nSparse Attention [67]: Self-attention has O(n2) time complex-\nity which becomes infeasible for large sequences. To speed\nup the computation, sparse attention [67] iteratively calculates\nattention in sliding windows for speed gains.\nFlash Attention [68]: Memory access is the major bottleneck\nin calculating attention using GPUs. To speed up, flash\nattention employs input tiling to minimize the memory reads\nand writes between the GPU hig